In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

import yaml
config_file = yaml.safe_load(open('../config.yaml', 'r'))

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import torch
from src.model.models_dsfno_3d import DSFNO
from src.dataloader.dataloader_3d import dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dsfno_model = DSFNO(in_channel=5, 
                    modes=config_file['dsfno']['modes'],
                    n_channels=config_file['dsfno']['n_channels'],
                    n_residual_blocks=config_file['dsfno']['n_residual_blocks'],
                    n_operator_blocks=config_file['dsfno']['n_operator_blocks'], 
                    apply_constraint=config_file['dsfno']['apply_constraint']).to(device)

In [ ]:
#max_samples = 30
dataset = dataset_sr()

In [ ]:
hr_state, lr_state, energy, mass = dataset[0]

In [ ]:
loss = torch.nn.MSELoss()

In [ ]:
hr_state = hr_state.numpy()

In [ ]:
import jax
import jax.numpy as jnp

# timing
from timeit import default_timer as timer

# jf1uids data structures
from jf1uids import SimulationConfig
from jf1uids import SimulationParams
from jf1uids.option_classes import WindConfig
from jf1uids.option_classes.simulation_config import BACKWARDS, OSHER, FORWARDS
# jf1uids setup functions
from jf1uids import get_helper_data
from jf1uids.fluid_equations.fluid import (
    construct_primitive_state,
    get_absolute_velocity,
    total_energy_from_primitives,
)

from jf1uids.fluid_equations import total_quantities as tq
from jf1uids import get_registered_variables
from jf1uids.option_classes.simulation_config import finalize_config

# turbulent ic setup
from jf1uids.initial_condition_generation.turb import create_turb_field

# main simulation function
from jf1uids import time_integration

# units
from jf1uids import CodeUnits
from astropy import units as u
import astropy.constants as c
from fractions import Fraction

import Pk_library as PKL


In [ ]:
print("👷 Setting up simulation...")

# simulation settings
gamma = float(Fraction(config_file["turbulent_sim"]["gamma"]))

wanted_rms = config_file["turbulent_sim"]["wanted_rms"] * u.km / u.s

# setup simulation config
config = SimulationConfig(
    runtime_debugging = config_file["turbulent_sim"]["runtime_debug"],
    first_order_fallback = config_file["turbulent_sim"]["first_order_fb"],
    progress_bar = config_file["turbulent_sim"]["progress_bar"],
    dimensionality = config_file["turbulent_sim"]["dimensionality"],
    num_ghost_cells = config_file["turbulent_sim"]["num_ghost_cells"],
    box_size = config_file["turbulent_sim"]["box_size"], 
    num_cells = config_file["turbulent_sim"]["num_cells"],
    fixed_timestep = config_file["turbulent_sim"]["fixed_timestep"],
    differentiation_mode = FORWARDS,
    return_snapshots = config_file["turbulent_sim"]["return_snapshots"],
    num_snapshots = config_file["turbulent_sim"]["num_snapshots"]
)
config = finalize_config(config, (5, 128, 128, 128))
helper_data = get_helper_data(config)
registered_variables = get_registered_variables(config)

primitive_state = jnp.array(hr_state)


In [ ]:
def get_energy_spectrum(primitive_state, config, registered_variables, gamma):
    """Calculate the total energy from the primitive state."""
    rho = primitive_state[registered_variables.density_index]
    u = get_absolute_velocity(primitive_state, config, registered_variables)
    p = primitive_state[registered_variables.pressure_index]
    energy = np.array(total_energy_from_primitives(rho, u, p, gamma), dtype=np.float32)
    pk_energy = PKL.Pk(delta=energy, BoxSize=1, axis=0, MAS="None", threads=6, verbose=False)
    return pk_energy


In [ ]:
def calculate_total_quantities(primitive_state, helper_data, gamma, config, registered_variables):
    total_e = tq.calculate_total_energy(
        primitive_state = primitive_state, 
        helper_data = helper_data, 
        gamma = gamma, 
        gravitational_constant = c.G.value,
        config = config,
        registered_variables = registered_variables)
    grav_e = tq.calculate_gravitational_energy(
        state=primitive_state,
        helper_data=helper_data,
        gravitational_constant=c.G.value,
        config=config,
        registered_variables=registered_variables,
    )
    internal_e = tq.calculate_internal_energy(
        state=primitive_state,
        helper_data=helper_data,
        gamma=gamma,
        config=config,
        registered_variables=registered_variables,
    )
    kinetic_e = tq.calculate_kinetic_energy(
        state=primitive_state,
        helper_data=helper_data,
        config=config,
        registered_variables=registered_variables,
    )
    total_mass = tq.calculate_total_mass(
    primitive_state = primitive_state, 
    helper_data = helper_data, 
    config = config)

    return total_e, grav_e, internal_e, kinetic_e, total_mass

In [ ]:
def internal_energy_spectrum(state, gamma, config, registered_variables):
    num_ghost_cells = config.num_ghost_cells
    p = state[registered_variables.pressure_index]

    if config.cosmic_ray_config.cosmic_rays:
        gamma_cr = 4/3
        p = p - state[registered_variables.cosmic_ray_n_index] ** gamma_cr

    internal_energy = p / (gamma - 1)

    # Remove ghost cells if necessary
    if config.dimensionality == 1:
        internal_energy = internal_energy[num_ghost_cells:-num_ghost_cells]
    else:
        slices = tuple(slice(num_ghost_cells, -num_ghost_cells) for _ in range(config.dimensionality))
        internal_energy = internal_energy[slices]

    internal_energy_np = np.array(internal_energy)

    fft_energy = np.fft.fftn(internal_energy_np)
    fft_energy_shifted = np.fft.fftshift(fft_energy)
    power_spectrum = np.abs(fft_energy_shifted)**2

    return power_spectrum

def calculate_kinetic_energy(state, helper_data, config, registered_variables):
    num_ghost_cells = config.num_ghost_cells

    rho = state[registered_variables.density_index]
    u = get_absolute_velocity(state, config, registered_variables)

    kinetic_energy = 0.5 * rho * u ** 2

    if config.dimensionality == 1:
        return jnp.sum(kinetic_energy[num_ghost_cells:-num_ghost_cells] * helper_data.cell_volumes[num_ghost_cells:-num_ghost_cells])
    else:
        return jnp.sum(kinetic_energy * config.grid_spacing**config.dimensionality)

def kinetic_energy_spectrum(state, helper_data, config, registered_variables):
    num_ghost_cells = config.num_ghost_cells
    rho = state[registered_variables.density_index]


    u = get_absolute_velocity(state, config, registered_variables)

    kinetic_energy = 0.5 * rho * u ** 2

    if config.dimensionality == 1:
        kinetic_energy =  kinetic_energy[num_ghost_cells:-num_ghost_cells] * helper_data.cell_volumes[num_ghost_cells:-num_ghost_cells]
    else:
         kinetic_energy =  kinetic_energy * config.grid_spacing**config.dimensionality

    kinetic_energy_np = np.array(kinetic_energy)

    fft_energy = np.fft.fftn(kinetic_energy_np)
    fft_energy_shifted = np.fft.fftshift(fft_energy)
    power_spectrum = np.abs(fft_energy_shifted)**2

    return power_spectrum

def mass_spectrum(
    state,
    helper_data,
    config,
):
    num_ghost_cells = config.num_ghost_cells

    if config.dimensionality == 1:
        mass = state[0, num_ghost_cells:-num_ghost_cells] * helper_data.cell_volumes[num_ghost_cells:-num_ghost_cells]
    else:
        slice_off_ghost_cells = (0,) + (slice(num_ghost_cells, -num_ghost_cells),) * config.dimensionality
        # note that here the box size is assumed to be the box size without the ghost cells
        mass = state[slice_off_ghost_cells] * config.box_size**config.dimensionality
    
    mass_np = np.array(mass)

    fft_mass = np.fft.fftn(mass_np)
    fft_mass_shifted = np.fft.fftshift(fft_mass)
    power_spectrum = np.abs(fft_mass_shifted)**2

    return power_spectrum

def radial_spectrum(power_spectrum):
    shape = power_spectrum.shape
    center = [s // 2 for s in shape]
    
    # Create coordinate grids
    z, y, x = np.indices(shape)
    k = np.sqrt((x - center[2])**2 + (y - center[1])**2 + (z - center[0])**2)
    k = k.astype(int)

    # Bin average
    k_max = int(np.max(k))
    spectrum = np.zeros(k_max + 1)
    counts = np.zeros(k_max + 1)
    
    for i in range(k_max + 1):
        mask = (k == i)
        spectrum[i] = power_spectrum[mask].sum()
        counts[i] = mask.sum()

    return spectrum / np.maximum(counts, 1)



In [ ]:
internal_e_spectrum = radial_spectrum(internal_energy_spectrum(    
    state = primitive_state, 
    gamma = gamma,
    config = config,
    registered_variables = registered_variables))

kinetic_e_spectrum = radial_spectrum(kinetic_energy_spectrum(    
    state=primitive_state, 
    helper_data=helper_data,
    config=config,
    registered_variables=registered_variables))

mass_spectrum = radial_spectrum(mass_spectrum(    
    state=primitive_state, 
    helper_data=helper_data,
    config=config))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (15, 5))
ax[0].scatter(range(0, len(internal_e_spectrum)), internal_e_spectrum)
ax[0].set_yscale('log')
ax[0].set_xscale('log')
ax[0].set_title('Internal energy')

ax[1].scatter(range(0, len(kinetic_e_spectrum)), kinetic_e_spectrum)
ax[1].set_yscale('log')
ax[1].set_xscale('log')
ax[1].set_title('Kinetic energy')

ax[2].scatter(range(0, len(mass_spectrum)), mass_spectrum)
ax[2].set_yscale('log')
ax[2].set_xscale('log')
ax[2].set_title('mass spectrum')

#ax[2].scatter(range(0, len(kinetic_e_spectrum)), internal_e_spectrum + kinetic_e_spectrum)
#ax[2].set_yscale('log')
#ax[2].set_title('kin + int spectrum')